In [1]:
import torch
from diffusers import DDPMPipeline

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [2]:
# Upload zip to Colab
from google.colab import drive
import shutil

drive.mount('/content/drive')
# shutil.copytree( # Move to Local memory
#     "/content/drive/MyDrive/ddpm_model",
#     "/content/ddpm_model"
# )

Mounted at /content/drive


In [3]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH = "./drive/MyDrive/ddpm_model"
SEED = 42

print(f"Using device: {DEVICE}")

Using device: cuda


In [10]:
pipeline = DDPMPipeline.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)
generator = torch.Generator(device=DEVICE).manual_seed(42)

print(pipeline.device)
print(next(pipeline.unet.parameters()).dtype)

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

cuda:0
torch.float16


In [ ]:
with torch.inference_mode():
    if DEVICE == "cuda":
        with torch.autocast("cuda", dtype=torch.float16):
            images = pipeline(
                batch_size=8,
                num_inference_steps=1000,   # try 40/60/100
                generator=generator,
                output_type="pil"
            ).images
    else:
        images = pipeline(
            batch_size=8,
            num_inference_steps=1000,
            generator=generator,
            output_type="pil"
        ).images

In [9]:
import os

output_dir = "./generated_images"
os.makedirs(output_dir, exist_ok=True)

for i, img in enumerate(images):
    img.save(os.path.join(output_dir, f"sample_{i+1}.png"))

print(f"Saved {len(images)} images to {output_dir}")

Saved 8 images to ./generated_images


The images have been saved to the `./generated_images` directory. You can now view them from the files pane on the left.